# Background

Author: Paul Inkenbrandt
Date: 11/1/2026

Purpose: Batch-run every footprint model in `fluxfootprints` (`ffp_xr`, `ffp`, `km`, `ls`,
`wang`) for multiple stations and save every result as a Zarr store.

For each station, each month in the analysis period, and each model, the notebook writes:

| Store | Written by | Models | Contents |
|---|---|---|---|
| `{station}/{model}/{station}_{model}_{yyyymm}_results.zarr` | `model.to_zarr()` | all | the model's full results Dataset (climatology, plus the per-timestep footprint for `ffp`/`ffp_xr`) |
| `{station}/{model}/{station}_{model}_{yyyymm}_summaries.zarr` | `export_summaries_zarr()` | `ffp`, `ffp_xr` | daily/monthly mean and ET-weighted footprints, georeferenced to local UTM |
| `{station}/{station}_monthly_climatology.zarr` | this notebook | all | one variable per model, dims `(time_monthly, y, x)`, every model regridded onto the same grid |

A `run_manifest.csv` in `OUT_DIR` lists every station/month/model run, its status, runtime,
and the stores it produced.

Things to know before running:

- **Only `ffp` and `ffp_xr` have a per-timestep footprint**, so only they get daily/monthly
  `summaries` stores. `km`, `ls` and `wang` only produce a climatology.
- **`ls` is Monte-Carlo** (cost ~ `n_particles x n_timesteps`); `LS_N_PARTICLES` is kept low
  by default. Drop `"ls"` from `MODEL_TYPES` for long periods if runtime is a problem.
- **`wang` only uses unstable rows** (`ol < 0`); months with too few are recorded as failed in
  the manifest rather than stopping the run.
- Models run month by month and are released after each month so memory stays bounded.

# Parameters and Libraries

In [ ]:
import gc
import json
import time
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import CRS, Transformer

sys.path.append("../../src")
from fluxfootprints import write_zarr
from fluxfootprints.ffp_daily_monthly_helper import (
    build_climatology,
    choose_utm_epsg,
    compute_aerodynamic_params,
    export_summaries_zarr,
    summarize_periods,
)

In [ ]:
# Stations to run. None = every station in STATIONS_GEOJSON that has an input file.
STATION_IDS = None   # e.g. ["US-UTW", "US-UTM", "US-UTE"]

STATIONS_GEOJSON = Path("../../data/station_config/stations.geojson")   # station lat/lon
DAT_PATH_TEMPLATE = "input_data/{sid}_footprint.parquet"
# Optional NLDAS ETo for ET weighting (same format as footprint_workflow_example.ipynb).
# Stations without this file fall back to the tower ET column below.
NLDAS_ET_PATH_TEMPLATE = "input_data/{sid}_nldas.csv"
TOWER_ET_COLUMN = "et_1_1_1"   # tower ET in mm/hr

OUT_DIR = Path("output/zarr_runs")

# Analysis period (include the time of day) and daytime hours, 8:00-20:00 follows Volk (2023)
START = "2021-06-01 00:00"
END   = "2021-07-31 23:30"
START_TIME = 8
END_TIME = 20
DATA_FREQ = "30min"   # tower data frequency, used to resample NLDAS ET

DX, DY = 10.0, 10.0   # output raster resolution [m]
DOMAIN = (-600.0, 600.0, -600.0, 600.0)

MODEL_TYPES = ["ffp_xr", "ffp", "km", "ls", "wang"]
SUMMARY_MODELS = {"ffp_xr", "ffp"}   # models with a per-timestep footprint
LS_N_PARTICLES = 5000                # ls_footprint_adapter default is 20000
MIN_VALID_TIMESTEPS = 10             # skip a month/model with fewer usable rows

WRITE_FULL_RESULTS = True   # per-month model.to_zarr(); includes the (time, x, y) stack for ffp/ffp_xr
WRITE_SUMMARIES = True      # per-month export_summaries_zarr() for SUMMARY_MODELS
WRITE_COMBINED = True       # per-station monthly climatology store, all models on one grid

# Column mapping shared by every model
COLUMNS = dict(
    ustar="ustar_1_1_1",
    ol="mo_length_1_1_1",
    umean="ws_1_1_1",
    sigmav="v_sigma_1_1_1",
    wind_dir="wd_1_1_1",
    zm="zm",
    z0="z0",
)
REQ_FIELDS = list(COLUMNS.values())

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Station coordinates from the station GeoJSON
with open(STATIONS_GEOJSON) as f:
    features = json.load(f)["features"]
station_coords = {
    feat["properties"]["Site ID"]: feat["geometry"]["coordinates"]   # [lon, lat]
    for feat in features
}

station_ids = STATION_IDS if STATION_IDS is not None else sorted(station_coords)
stations = []
for sid in station_ids:
    dat_path = Path(DAT_PATH_TEMPLATE.format(sid=sid))
    if sid not in station_coords:
        print(f"{sid}: no coordinates in {STATIONS_GEOJSON.name}, skipping")
        continue
    if not dat_path.exists():
        if STATION_IDS is not None:
            print(f"{sid}: input file {dat_path} not found, skipping")
        continue
    lon, lat = station_coords[sid]
    nldas_path = Path(NLDAS_ET_PATH_TEMPLATE.format(sid=sid))
    stations.append({
        "station": sid,
        "lat": lat,
        "lon": lon,
        "dat_path": dat_path,
        "et_source": "nldas" if nldas_path.exists() else "tower",
    })

stations_df = pd.DataFrame(stations)
assert len(stations_df) > 0, "No stations with input data found - check STATION_IDS and DAT_PATH_TEMPLATE"
stations_df

## Helper Functions

- `load_station` reads one station's eddy covariance data, computes `z0`/`zm`, attaches an
  `et_weight` column (NLDAS ETo if available, otherwise tower ET), and subsets to the analysis
  period and daytime hours.
- `regrid` puts a model climatology onto the common `DOMAIN`/`DX`/`DY` grid. `ffp`, `ffp_xr`
  and `km` already match it; `ls` (histogram bin centres) and `wang` (own x axis) don't. Cells
  outside a model's native grid are filled with 0 and the result is renormalized to integrate
  to 1 (same approach as
  [footprint_model_comparison_all_sites.ipynb](footprint_model_comparison_all_sites.ipynb)).
- `write_combined_store` writes the per-station store holding every model's monthly
  climatology, with UTM coordinates and a CF `spatial_ref` like `export_summaries_zarr`.

In [ ]:
common_x = np.arange(DOMAIN[0], DOMAIN[1] + DX, DX)
common_y = np.arange(DOMAIN[2], DOMAIN[3] + DY, DY)
cell_area = DX * DY


def load_station(row) -> pd.DataFrame:
    df = pd.read_parquet(row.dat_path).sort_index()
    df = compute_aerodynamic_params(
        df,
        inst_height="measurement_height",
        crop_height="crop_height_m",
        veg_type="alfalfa",
    )

    if row.et_source == "nldas":
        nldas = pd.read_csv(
            NLDAS_ET_PATH_TEMPLATE.format(sid=row.station),
            parse_dates=["timestamp_local"],
            index_col="timestamp_local",
        )
        nldas = nldas["eto_grass"].sort_index().resample(DATA_FREQ).ffill()
        df["et_weight"] = nldas.reindex(df.index)
    elif TOWER_ET_COLUMN in df.columns:
        df["et_weight"] = df[TOWER_ET_COLUMN]
    else:
        df["et_weight"] = np.nan

    df = df.loc[START:END]
    return df[(df.index.hour >= START_TIME) & (df.index.hour <= END_TIME)]


def regrid(fclim: xr.DataArray) -> np.ndarray:
    """Climatology on the common grid, dims (y, x), integrating to 1."""
    out = fclim.transpose("x", "y").interp(
        x=common_x, y=common_y, method="linear", kwargs={"fill_value": 0.0}
    ).fillna(0.0)
    arr = np.clip(out.values, 0.0, None)   # interpolation can create tiny negative ripples
    total = arr.sum() * cell_area
    if total > 0:
        arr = arr / total
    return arr.T


def write_combined_store(sid, lat, lon, climatologies, store) -> Path:
    """climatologies: {model_type: {month_start (Timestamp): (y, x) array}}"""
    months = sorted({m for per_model in climatologies.values() for m in per_model})
    data_vars = {}
    for model_type, per_month in climatologies.items():
        stack = np.full((len(months), len(common_y), len(common_x)), np.nan, dtype="float32")
        for i, month in enumerate(months):
            if month in per_month:
                stack[i] = per_month[month]
        data_vars[model_type] = xr.DataArray(
            stack,
            dims=("time_monthly", "y", "x"),
            attrs={
                "long_name": f"Monthly {model_type} footprint climatology (regridded)",
                "units": "m-2",
                "grid_mapping": "spatial_ref",
            },
        )

    utm_epsg = choose_utm_epsg(lon, lat)
    utm_crs = CRS.from_epsg(utm_epsg)
    x0, y0 = Transformer.from_crs(4326, utm_crs, always_xy=True).transform(lon, lat)

    ds = xr.Dataset(
        data_vars,
        coords={
            "time_monthly": pd.DatetimeIndex(months),
            "x": ("x", common_x, {"units": "m", "long_name": "Distance east of tower"}),
            "y": ("y", common_y, {"units": "m", "long_name": "Distance north of tower"}),
            "easting": ("x", common_x + x0, {"units": "m", "standard_name": "projection_x_coordinate"}),
            "northing": ("y", common_y + y0, {"units": "m", "standard_name": "projection_y_coordinate"}),
            "spatial_ref": xr.DataArray(
                np.int32(0),
                attrs={**utm_crs.to_cf(), "crs_wkt": utm_crs.to_wkt(), "epsg_code": f"EPSG:{utm_epsg}"},
            ),
        },
        attrs={
            "title": f"{sid} monthly footprint climatology by model",
            "station": sid,
            "station_lat": float(lat),
            "station_lon": float(lon),
            "tower_easting": float(x0),
            "tower_northing": float(y0),
            "crs": f"EPSG:{utm_epsg}",
            "models": list(climatologies),
            "dx": DX,
            "dy": DY,
            "daytime_hours": f"{START_TIME}-{END_TIME}",
            "Conventions": "CF-1.8",
            "source": "fluxfootprints",
        },
    )
    write_zarr(ds, store)
    return Path(store)

## Run Every Model for Every Station

Loop order is station -> month -> model. Every run is wrapped in `try`/`except`, so one model
failing for one month (e.g. `wang` with no unstable rows) is recorded in the manifest and the
loop moves on. Existing stores are overwritten.

In [ ]:
manifest = []         # one row per station/month/model
combined_stores = {}  # station -> path of the combined monthly climatology store
t_start = time.perf_counter()

for row in stations_df.itertuples():
    sid = row.station
    print(f"\n=== {sid} ({row.lat:.4f}, {row.lon:.4f}), ET weights from {row.et_source} ===")
    df_station = load_station(row)
    if df_station.empty:
        print(f"  no data between {START} and {END}, skipping")
        continue

    station_dir = OUT_DIR / sid
    climatologies = {m: {} for m in MODEL_TYPES}   # model -> {month_start: (y, x) array}

    for period in pd.period_range(df_station.index.min(), df_station.index.max(), freq="M"):
        label = str(period)
        yyyymm = label.replace("-", "")
        df_mon = df_station.loc[label].dropna(subset=REQ_FIELDS)
        n_unstable = int((df_mon[COLUMNS["ol"]] < 0).sum())
        print(f"  {label}: {len(df_mon)} valid rows ({n_unstable} unstable)")

        for model_type in MODEL_TYPES:
            rec = {"station": sid, "month": label, "model": model_type,
                   "n_input_rows": len(df_mon), "status": "ok",
                   "results_store": None, "summaries_store": None}
            model_dir = station_dir / model_type
            model_dir.mkdir(parents=True, exist_ok=True)
            model = summaries = None
            t0 = time.perf_counter()
            try:
                if len(df_mon) < MIN_VALID_TIMESTEPS:
                    raise ValueError(f"fewer than {MIN_VALID_TIMESTEPS} valid rows")

                model = build_climatology(
                    df_mon,
                    model_type=model_type,
                    **COLUMNS,
                    h=2000.0,
                    dx=DX,
                    dy=DY,
                    domain=DOMAIN,
                    verbosity=0,
                    **({"n_particles": LS_N_PARTICLES} if model_type == "ls" else {}),
                )
                rec["n_timesteps"] = getattr(model, "ts_len", None) or model.results.attrs.get("n_timesteps")

                if WRITE_FULL_RESULTS:
                    store = model_dir / f"{sid}_{model_type}_{yyyymm}_results.zarr"
                    model.to_zarr(store)
                    rec["results_store"] = str(store)

                if WRITE_SUMMARIES and model_type in SUMMARY_MODELS:
                    has_et = df_mon["et_weight"].notna().any()
                    summaries = summarize_periods(
                        model,
                        df_mon,
                        et_source="et_weight" if has_et else None,
                        is_le=False,
                        calc_et_weighted=has_et,
                        daily=True,
                        monthly=True,
                        normalize_each_time=True,
                    )
                    store = model_dir / f"{sid}_{model_type}_{yyyymm}_summaries.zarr"
                    export_summaries_zarr(
                        model, summaries,
                        station_lat=row.lat, station_lon=row.lon,
                        store=store,
                    )
                    rec["summaries_store"] = str(store)
                    cov = summaries.monthly_domain_coverage
                    rec["mean_coverage_pct"] = float(cov["mean_coverage_pct"].iloc[0])

                climatologies[model_type][period.to_timestamp()] = regrid(model.get_footprint_climatology())

            except Exception as exc:
                rec["status"] = f"failed: {exc}"
            finally:
                rec["runtime_s"] = round(time.perf_counter() - t0, 2)
                manifest.append(rec)
                print(f"    {model_type:7s} {rec['status'][:70]:70s} {rec['runtime_s']:7.1f}s")
                del model, summaries
                gc.collect()

    climatologies = {m: c for m, c in climatologies.items() if c}
    if WRITE_COMBINED and climatologies:
        store = station_dir / f"{sid}_monthly_climatology.zarr"
        combined_stores[sid] = write_combined_store(sid, row.lat, row.lon, climatologies, store)
        print(f"  combined store: {store}")

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / "run_manifest.csv", index=False)
print(f"\nFinished in {time.perf_counter() - t_start:.0f}s; manifest at {OUT_DIR / 'run_manifest.csv'}")

## Review the Run

In [ ]:
ok = manifest_df["status"] == "ok"
print(f"{ok.sum()} of {len(manifest_df)} station/month/model runs succeeded")
manifest_df.pivot_table(index=["station", "month"], columns="model", values="status",
                        aggfunc=lambda s: "ok" if (s == "ok").all() else "failed")

In [ ]:
# Failure reasons, if any
manifest_df.loc[~ok, ["station", "month", "model", "n_input_rows", "status"]]

In [ ]:
# Runtime per model, summed over stations and months
manifest_df.groupby("model")["runtime_s"].sum().reindex(MODEL_TYPES).round(1)

## Check the Zarr Output

Open one station's combined store lazily with `xarray.open_zarr` and plot every model's
climatology for the first month. Because all models share one grid, the variables can be
compared or differenced directly. The `easting`/`northing` coordinates and the `spatial_ref`
variable place the grid in UTM (e.g. `ds.rio.write_crs(ds.attrs["crs"])` with rioxarray).

In [ ]:
sid = next(iter(combined_stores))
ds = xr.open_zarr(combined_stores[sid])
ds

In [ ]:
month = ds["time_monthly"].values[0]
models_in_store = list(ds.data_vars)
vmax = max(float(ds[m].sel(time_monthly=month).max()) for m in models_in_store)

fig, axes = plt.subplots(1, len(models_in_store), figsize=(4.2 * len(models_in_store), 4.2), squeeze=False)
axes = axes[0]
for ax, m in zip(axes, models_in_store):
    im = ax.pcolormesh(ds["x"], ds["y"], ds[m].sel(time_monthly=month), cmap="viridis",
                       vmin=0, vmax=vmax, shading="auto")
    ax.plot(0, 0, marker="^", color="white", markeredgecolor="black", markersize=9, zorder=5)
    ax.set_title(m)
    ax.set_xlabel("x [m]")
    ax.set_aspect("equal")
axes[0].set_ylabel("y [m]")
fig.colorbar(im, ax=axes.tolist(), shrink=0.85, label="Footprint density [m$^{-2}$]")
fig.suptitle(f"{sid} footprint climatology, {pd.Timestamp(month):%Y-%m}", y=1.03)
plt.show()

In [ ]:
# A per-month summaries store (ffp/ffp_xr): daily and monthly layers on separate time axes
summary_stores = manifest_df["summaries_store"].dropna()
if len(summary_stores):
    xr.open_zarr(summary_stores.iloc[0])

## Notes & Caveats

- **Re-running overwrites** every store for the stations and months processed. Point `OUT_DIR`
  somewhere new to keep an earlier run.
- **Combined store vs. per-month stores.** The combined store holds regridded, renormalized
  climatologies, which suits model and month comparisons. Use the per-month `results`/
  `summaries` stores when you need each model's native grid or the daily layers.
- **Summaries use `normalize_each_time=True`**, as in
  [footprint_workflow_example.ipynb](footprint_workflow_example.ipynb), so daily/monthly
  values there are per-cell weights summing to 1, not densities.
- **ET weighting** uses NLDAS ETo where `NLDAS_ET_PATH_TEMPLATE` exists and tower ET
  (`TOWER_ET_COLUMN`) otherwise; the source per station is in `stations_df`. Without either,
  the ET-weighted layers are left out of the summaries stores.
- **`ls` is stochastic**, so its stores change slightly between runs.
- **Store size.** `WRITE_FULL_RESULTS` keeps the per-timestep `(time, x, y)` stack for
  `ffp`/`ffp_xr`, which dominates disk use over long periods. Set it to `False` if the
  summaries and combined stores are enough.